# Castlevania RL Colab Trainer
## Cell 1: Environment & CUDA Hardware Check
Verify NVIDIA GPU hardware availability, query PyTorch CUDA specifications, and install required system and Python dependencies.

In [ ]:
# Check GPU availability and print hardware specifications
import torch
print("PyTorch Version:", torch.__version__)
gpu_available = torch.cuda.is_available()
print("CUDA Available:", gpu_available)
if gpu_available:
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No CUDA GPU detected. Please enable T4 GPU acceleration in Colab Runtime settings.")

# Install system dependencies and Python packages for stable-retro and headless display
!apt-get update && apt-get install -y ffmpeg python3-opengl xvfb libsdl2-dev
!pip install stable-retro gymnasium torch matplotlib pyvirtualdisplay


## Cell 2: Virtual Display Setup (Headless OpenGL)
Initialize a virtual framebuffer (Xvfb / pyvirtualdisplay) so stable-retro can render NES screen frames headlessly without requiring a physical display.

In [ ]:
import os
from pyvirtualdisplay import Display

print("Starting virtual display (Xvfb)...")
display = Display(visible=0, size=(1024, 768))
display.start()
print("Virtual display successfully started!")


## Cell 3: Repository Setup & ROM Import
Set the repository working directory, locate `Castlevania (USA).nes` (or prompt for upload if missing), and register the ROM with `stable-retro`.

In [ ]:
import os
import sys
import glob
import retro

# Set repository root working directory
repo_root = "/content/Game" if os.path.exists("/content/Game") else os.getcwd()
os.chdir(repo_root)
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
print(f"Current working directory: {os.getcwd()}")

rom_filename = "Castlevania (USA).nes"
rom_path = os.path.join(repo_root, rom_filename)

if not os.path.exists(rom_path):
    # Check inside roms/ directory
    alt_rom_path = os.path.join(repo_root, "roms", rom_filename)
    if os.path.exists(alt_rom_path):
        import shutil
        shutil.copy(alt_rom_path, rom_path)
        print(f"Copied ROM from {alt_rom_path} to {rom_path}")

if not os.path.exists(rom_path):
    print(f"{rom_filename} not found locally. Prompting for upload...")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if fn != rom_filename:
                os.rename(fn, rom_path)
    except Exception as e:
        print(f"Upload prompt skipped or unavailable: {e}")

if os.path.exists(rom_path):
    print(f"Importing {rom_path} into stable-retro...")
    retro.data.merge_into_master_python_where_necessary([rom_path])
    print("ROM imported and registered successfully!")
else:
    print(f"WARNING: {rom_filename} could not be located.")


## Cell 4: Google Drive Integration (Persistent Checkpoint Storage)
Mount Google Drive and configure dedicated checkpoint storage folder `/content/drive/MyDrive/Castlevania_RL_Checkpoints` for persistent model synchronization.

In [ ]:
import os
from google.colab import drive

drive_mount_path = "/content/drive"
gdrive_checkpoint_dir = "/content/drive/MyDrive/Castlevania_RL_Checkpoints"

try:
    drive.mount(drive_mount_path, force_remount=True)
    os.makedirs(gdrive_checkpoint_dir, exist_ok=True)
    print(f"Google Drive mounted. Checkpoints folder ready at: {gdrive_checkpoint_dir}")
except Exception as e:
    print(f"Google Drive mounting skipped or unavailable in current environment: {e}")


## Cell 5: Checkpoint Resolution & Warm Start
Restore historical target checkpoint (`ppo_agent_ep5000.pt` or `best_ppo_agent_dist_*.pt`) and sync Google Drive checkpoints if runtime storage was reset.

In [ ]:
import os
import glob
import shutil

local_ckpt_dir = os.path.abspath("checkpoints")
os.makedirs(local_ckpt_dir, exist_ok=True)
gdrive_ckpt_dir = "/content/drive/MyDrive/Castlevania_RL_Checkpoints"

# Copy latest checkpoints from Google Drive if available
if os.path.exists(gdrive_ckpt_dir):
    drive_ckpts = glob.glob(os.path.join(gdrive_ckpt_dir, "*.pt"))
    if drive_ckpts:
        drive_ckpts.sort(key=os.path.getmtime, reverse=True)
        for ckpt in drive_ckpts:
            dest_path = os.path.join(local_ckpt_dir, os.path.basename(ckpt))
            if not os.path.exists(dest_path):
                shutil.copy(ckpt, dest_path)
                print(f"Restored checkpoint from Drive: {os.path.basename(ckpt)}")

# Find target checkpoint
default_target = os.path.join(local_ckpt_dir, "ppo_agent_ep5000.pt")
best_dist_pattern = os.path.join(local_ckpt_dir, "best_ppo_agent_dist_*.pt")
best_dist_files = glob.glob(best_dist_pattern)

selected_checkpoint = None
if best_dist_files:
    best_dist_files.sort(key=os.path.getmtime, reverse=True)
    selected_checkpoint = best_dist_files[0]
elif os.path.exists(default_target):
    selected_checkpoint = default_target
else:
    local_pt_files = glob.glob(os.path.join(local_ckpt_dir, "*.pt"))
    if local_pt_files:
        local_pt_files.sort(key=os.path.getmtime, reverse=True)
        selected_checkpoint = local_pt_files[0]

if selected_checkpoint:
    print(f"Target checkpoint resolved for warm start: {selected_checkpoint}")
else:
    print("No existing checkpoint found. Training will initialize from scratch.")


## Cell 6: Training Execution Loop (24/7 Watchdog Pipeline)
Execute the 24/7 watchdog training supervisor under `xvfb-run` with CUDA acceleration and stream live telemetry directly to cell output.

In [ ]:
import os
import sys
import shutil
import time
import glob

rom_file = "./Castlevania (USA).nes"
print("Launching watchdog supervisor pipeline...")

# Execute watchdog supervisor process with continuous telemetry output
!xvfb-run -a python scripts/watchdog.py --use-retro --rom-path "./Castlevania (USA).nes" --cuda

# Sync newly generated checkpoints back to Google Drive backup storage
gdrive_ckpt_dir = "/content/drive/MyDrive/Castlevania_RL_Checkpoints"
local_ckpt_dir = os.path.abspath("checkpoints")
if os.path.exists(gdrive_ckpt_dir) and os.path.exists(local_ckpt_dir):
    local_ckpts = glob.glob(os.path.join(local_ckpt_dir, "*.pt"))
    for ckpt in local_ckpts:
        dest_path = os.path.join(gdrive_ckpt_dir, os.path.basename(ckpt))
        shutil.copy(ckpt, dest_path)
        print(f"Backed up checkpoint to Google Drive: {os.path.basename(ckpt)}")
